# AI Forex Bot v2 — Training di Google Colab
Notebook untuk training model dengan data 7 tahun di Google Colab

## Persiapan:
1. Upload `.env` ke `MyDrive/forex-bot/.env`
2. (Opsional) Upload folder `models/` ke `MyDrive/forex-bot/models/` jika lanjutkan model lama

Data sudah termasuk dalam repo (`forex_data_7years.zip`), **tidak perlu upload data**.

---
## 1. Mount Google Drive
(hanya untuk .env dan backup model)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 2. Clone Repository

In [ ]:
!git clone https://github.com/snooters/bot-ai-forex-v2.git
%cd bot-ai-forex-v2

---
## 3. Setup .env & Extract Data

In [ ]:
import shutil, os, zipfile
from pathlib import Path

DRIVE_BASE = '/content/drive/MyDrive/forex-bot'
HIST_DIR = 'data/historical'

# ==========================================
# 3a. Copy .env dari Drive
# ==========================================
env_src = f'{DRIVE_BASE}/.env'
if os.path.exists(env_src):
    shutil.copy(env_src, '.env')
    print('\u2713 .env copied from Drive')
else:
    print('\u26a0 .env not found in Drive. Pakai .env.example sebagai fallback')
    shutil.copy('.env.example', '.env')

# ==========================================
# 3b. Extract zip data (sudah di repo)
# ==========================================
zip_path = 'forex_data_7years.zip'
if os.path.exists(zip_path):
    print('Extracting forex_data_7years.zip...')
    os.makedirs(HIST_DIR, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(HIST_DIR)
    print('\u2713 Extracted!')
else:
    print('\u26a0 forex_data_7years.zip not found in repo!')

# ==========================================
# 3c. Generate missing timeframes (H1, H4)
# ==========================================
print('\n--- Generating missing timeframes ---')
!python utils/generate_timeframes.py

# ==========================================
# 3d. Copy existing models (opsional)
# ==========================================
models_src = f'{DRIVE_BASE}/models'
if os.path.exists(models_src):
    !cp -r "{models_src}/"* models/
    print('\u2713 Existing models restored from Drive')
else:
    print('\u2022 No existing models found (starting fresh)')

print('\n\u2705 Setup selesai')

---
## 4. Install Dependencies

In [ ]:
!pip install -r requirements.txt
print('\n\u2705 Dependencies installed')

---
## 5. Verifikasi Data
Pastikan data parquet terbaca dengan benar.

In [ ]:
import pandas as pd
from pathlib import Path

data_dir = Path('data/historical')
if data_dir.exists():
    for parquet_file in sorted(data_dir.rglob('*.parquet')):
        df = pd.read_parquet(parquet_file)
        print(f'{parquet_file.name}: {len(df):,} rows | '
              f'{df["time"].min()} \u2192 {df["time"].max()}')
else:
    print('\u26a0 Folder data/historical tidak ditemukan!')

---
## 6. Jalankan Training

Parameter:
- `--from-storage`: pakai data parquet
- `--all`: training semua timeframe (M5, M15, M30, H1, H4)
- `--force`: bypass market check
- `--days 365`: 1 tahun data

\u23f1 Estimasi: 15-30 menit (tergantung GPU Colab)

In [ ]:
# Load .env config
from dotenv import load_dotenv
load_dotenv()

import os
print(f'HISTORICAL_YEARS={os.getenv("HISTORICAL_YEARS", "not set")}')
print(f'ROLLING_TRAINING_WINDOW_DAYS={os.getenv("ROLLING_TRAINING_WINDOW_DAYS", "not set")}')

print('\n\U0001f680 Mulai training...\n')

In [ ]:
!python main.py train --from-storage --all --force --days 365

---
## 7. Backup Model ke Google Drive

In [ ]:
models_drive = '/content/drive/MyDrive/forex-bot/models'
os.makedirs(models_drive, exist_ok=True)

!cp -r models/* "{models_drive}/"

if os.path.exists('models/retrain_counter.json'):
    shutil.copy('models/retrain_counter.json', models_drive)

print('\n\u2705 Models saved to Google Drive')
print(f'\U0001f4c1 {models_drive}')

---
## 8. Ringkasan

In [ ]:
from pathlib import Path
import json

print('\U0001f4ca Model versions:')
for model_dir in sorted(Path('models').glob('model_*')):
    if model_dir.is_dir():
        metadata = model_dir / 'metadata.json'
        perf = model_dir / 'performance.json'
        if metadata.exists():
            m = json.loads(metadata.read_text())
            print(f'  {m["version"]:20s} | created: {m["created_at"][:19]}')
        if perf.exists():
            p = json.loads(perf.read_text())
            print(f'  {"":>20s} | OOS: WR={p.get("win_rate","-")} PF={p.get("profit_factor","-")} Grade={p.get("grade","-")}')

print('\n\u2705 Training selesai!')
print('   • Copy models/* dari Google Drive ke project lokal')
print('   • Jalankan: python main.py live')